# 第 7 週 Part B：診所掛號系統（個人作品）

**規則：** 個人獨立完成；可以使用任何 AI 工具；**禁止與其他同學討論**。

**流程（照順序做）：**
1. 在下一格寫好**規格書**（可以請 AI 幫忙寫初稿，但你要讀過並修改成你要的樣子）。
2. 把規格書交給 AI，請它依照規格產生程式，貼到下面對應的格子。
3. 執行**測試清單**，看懂每一句的 AI 原始回覆與系統解讀。
4. 照最後一格的**交件前檢查清單**交件。

**系統需求（簡化版）：** 科別只有內科、外科、牙科，每科每天有固定名額。功能有掛號、取消、查詢剩餘名額。
🔴 **安全規定：系統不可以提供任何醫療建議或根據症狀推薦科別。** 使用者沒說出科別時，請他直接說要掛哪一科，或洽詢櫃台。

# 📋 規格書（請直接在這一格修改）

學號：411312422　姓名：石詠澤

## 1. 系統目的（一句話）
讓民眾能透過自然語言對話，輕鬆完成門診掛號、取消掛號與剩餘名額查詢。

2. 使用者會怎麼說（至少 5 句）
## 2. 使用者會怎麼說（至少 5 句）
1.我要掛號
2.要取消掛號
3.醫師還有沒有掛號名額
4.營業時間到幾點
5.還可以預約嗎

## 3. 資料表（最多 2 張）
| 表名 | 欄位 | 型態 | 說明 |
|科別名額表departments|dept_name|VARCHAR|科別名稱（主鍵，例：內科、外科、牙科）|
|科別名額表departments|daily_limit|INT|每日總名額上限（預設 5）|
|科別名額表departments|dregistered_count|INT|今日已掛號人數（預設 0）|



|掛號紀錄表 (registrations) | id | INT | 紀錄流水號（自增主鍵） |
|掛號紀錄表 (registrations) | patient_name | VARCHAR | 病患姓名 |
|掛號紀錄表 (registrations) | dept_name | VARCHAR | 欲掛號科別 |
|掛號紀錄表 (registrations) | status | VARCHAR | 掛號狀態（已掛號 / 已取消） |



初始資料：

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,對象`（例：`掛號,內科`）
- 動作只能是：掛號、取消、查詢、諮詢門診、未知
- 對象只能是：內科、外科、牙科、未指定、未知
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：絕對不可以根據使用者描述的症狀提供任何醫療建議、診斷或推薦科別。

絕對不可以回應非內科、外科、牙科以外的門診科別。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|

| 查詢名額 | 判斷動作為「查詢」 | 顯示指定科別或所有科別的剩餘名額。 | 無異動 |
| 門診掛號 | 判斷動作為「掛號」，且該科剩餘名額 > 0 | 提示「掛號成功！」。 | departments 的 registered_count + 1；registrations 新增筆數 |
| 門診掛號（額滿） | 判斷動作為「掛號」，但該科剩餘名額 = 0 | 提示「該科今日名額已滿，無法掛號。」 | 無異動 |
| 取消掛號 | 判斷動作為「取消」，且存在有效掛號紀錄 | 提示「取消掛號成功！」。 | departments 的 registered_count - 1；registrations 狀態改為 已取消 |
| 安全引導 | 使用者詢問症狀或未說明科別 | 提示「本系統不提供醫療診斷或推薦科別，請直接說明要掛哪一科或洽詢現場櫃檯。」 | 無異動 |



## 6. 例外情況（至少 3 個）
1.未指定科別：使用者僅說「我要掛號」，未提及內科、外科或牙科，系統提示請使用者說明具體科別。
2.名額已滿：欲掛號之科別名額已達到上限（5人），系統自動拒絕掛號並提示額滿。
3.詢問醫療建議/症狀診斷：使用者描述自身病況並詢問科別（如「頭痛要看哪科」），觸發安全規定，系統拒絕推薦並請使用者親洽櫃檯或直接指定科別。
4.重複取消或查無紀錄：使用者欲取消沒有掛號過或已取消的紀錄，系統提示查無掛號資料。
## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 |  |  |#,類型,使用者說的話,"預期 AI 解析 (動作,對象)",預期系統結果
1,正常,我想掛號內科,"掛號,內科",掛號成功，內科剩餘名額減 1
2,正常,請幫我取消牙科掛號,"取消,牙科",取消成功，牙科剩餘名額加 1
3,正常,請問外科還有名額嗎？,"查詢,外科",顯示外科目前剩餘名額
4,正常,目前各科還剩下多少名額？,"查詢,未指定",顯示內科、外科、牙科三科的剩餘名額
5,錯誤/安全,我最近肚子很痛，應該掛哪一科？,"諮詢門診,未指定",拒絕提供診斷，提示直接說明科別或洽詢現場櫃檯
6,錯誤,我想預約眼科門診,"掛號,未知",提示本院僅提供內科、外科、牙科服務
7,邊界,我要掛號,"掛號,未指定",提示使用者說出欲掛號的具體科別
8,邊界,（當內科已掛滿 5 人時）我要掛內科,"掛號,內科",系統提示「內科今日名額已滿」，拒絕新增掛號

## 🤖 可以使用的 prompt

**① 請 AI 寫規格書初稿**
```
我要做一個「診所掛號系統」的課堂小作品（Google Colab + SQLite + Groq + Gradio）。
請依照下面的規格範本，幫我寫出一份規格書初稿。要求：
1. 科別只有：內科、外科、牙科，每科每天有固定名額。
2. 功能：掛號、取消掛號、查詢某科剩餘名額。
3. 資料表不超過 2 張。
4. AI 只負責判斷「動作」與「科別」。如果使用者只描述症狀、沒有說出科別，AI 一律回答未知；系統絕對不可以提供任何醫療建議或推薦科別。
5. 測試清單至少 8 句，要包含正常、錯誤、邊界三種情況，並寫出預期結果。
規格範本如下：
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫一份 Google Colab 程式，並遵守以下要求：
1. 第 1 格（安裝套件與讀取金鑰）我已經有了，請從第 2 格開始寫，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) 測試清單 (6) 功能函式 (7) 主流程與完整流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經定義好 client（Groq）與 MODEL，也已經 import 了 sqlite3、pandas as pd、display、gradio as gr，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 temperature=0，並用 try/except 包起來，失敗時回傳「未知」，不可讓程式當掉。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」。
7. fetchone() 之後一定要先檢查是否為 None。
8. 資料庫連線用 try/finally 確保關閉。
9. 測試清單的格式為 (使用者說的話, 預期動作, 預期科別)，跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 只用 sqlite3、groq、gradio、pandas，不要用其他套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [1]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('SDDA_Lecture05'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [2]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
import sqlite3

DB_NAME = "hospital.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    try:
        # 4. 每次執行都先 DROP TABLE 避免殘留舊資料
        cursor.execute("DROP TABLE IF EXISTS registrations")
        cursor.execute("DROP TABLE IF EXISTS departments")

        # 建立科別名額表
        cursor.execute('''
            CREATE TABLE departments (
                dept_name TEXT PRIMARY KEY,
                daily_limit INTEGER NOT NULL,
                registered_count INTEGER NOT NULL
            )
        ''')

        # 建立掛號紀錄表
        cursor.execute('''
            CREATE TABLE registrations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                patient_name TEXT NOT NULL,
                dept_name TEXT NOT NULL,
                status TEXT NOT NULL
            )
        ''')

        # 寫入初始資料（內科、外科、牙科，名額各 5 人，初始已掛 0 人）
        departments_data = [
            ("內科", 5, 0),
            ("外科", 5, 0),
            ("牙科", 5, 0)
        ]
        # 5. 使用 ? 參數化查詢，避免 SQL Injection 攻擊
        cursor.executemany("INSERT INTO departments VALUES (?, ?, ?)", departments_data)

        conn.commit()
        print("✅ 資料庫初始化完成！科別名額表與掛號紀錄表已重建。")
    finally:
        # 8. 使用 try/finally 確保資料庫連線必定會關閉
        conn.close()

# 執行初始化
init_db()

✅ 資料庫初始化完成！科別名額表與掛號紀錄表已重建。


In [4]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
import sqlite3
import pandas as pd
from IPython.display import display

def show_tables():
    conn = sqlite3.connect(DB_NAME)
    try:
        print("📊 === 科別名額表 (departments) ===")
        df_dept = pd.read_sql_query("SELECT * FROM departments", conn)
        display(df_dept)

        print("\n📋 === 掛號紀錄表 (registrations) ===")
        df_reg = pd.read_sql_query("SELECT * FROM registrations", conn)
        display(df_reg)
    finally:
        conn.close()

# 測試顯示當前資料表
show_tables()

📊 === 科別名額表 (departments) ===


,dept_name,daily_limit,registered_count
0,內科,5,0
1,外科,5,0
2,牙科,5,0



📋 === 掛號紀錄表 (registrations) ===


,id,patient_name,dept_name,status


In [5]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 科別, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
ALLOWED_ACTIONS = ["掛號", "取消", "查詢", "諮詢門診", "未知"]
ALLOWED_DEPTS = ["內科", "外科", "牙科", "未指定", "未知"]

def parse_user_input(user_text):
    """
    呼叫 Groq 將使用者語意轉換為 (動作, 對象)
    """
    prompt = f"""你是一個醫院掛號系統的 NLP 語意解析器。
請分析使用者的輸入，並嚴格只輸出格式：「動作,對象」。

【動作只能是】：掛號、取消、查詢、諮詢門診、未知
【對象只能是】：內科、外科、牙科、未指定、未知

【嚴格安全規定】：
1. 當使用者描述症狀、詢問疾病診斷或問「要看哪科」時，動作必須輸出「諮詢門診」，對象輸出「未指定」。
2. 絕對不可以在對象中回答其他非內科、外科、牙科的科別。

使用者輸入："{user_text}"
輸出格式範例：掛號,內科
請只輸出「動作,對象」，不要包含任何額外說明或標點符號。"""

    # 3. 呼叫 Groq 使用 temperature=0，並用 try/except 包裹防止當掉
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw_reply = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ API 呼叫失敗: {e}")
        raw_reply = "未知,未知"

    # 6. 不可直接相信 AI 的回覆：採用允許清單 (Whitelist) 進行檢查
    parsed_action = "未知"
    parsed_dept = "未知"

    if "," in raw_reply:
        parts = raw_reply.split(",", 1)
        action_candidate = parts[0].strip()
        dept_candidate = parts[1].strip()

        if action_candidate in ALLOWED_ACTIONS:
            parsed_action = action_candidate
        if dept_candidate in ALLOWED_DEPTS:
            parsed_dept = dept_candidate

    return raw_reply, parsed_action, parsed_dept

In [7]:
# ============================================================
# 第 5 格：測試清單
# 提醒：至少 8 句，要與規格書第 7 節一致
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）test_cases = [
test_cases = [
    ("我想掛號內科", "掛號", "內科"),
    ("請幫我取消牙科掛號", "取消", "牙科"),
    ("請問外科還有名額嗎？", "查詢", "外科"),
    ("目前各科還剩下多少名額？", "查詢", "未指定"),
    ("我最近肚子很痛，應該掛哪一科？", "諮詢門診", "未指定"),
    ("我想預約眼科門診", "掛號", "未知"),
    ("我要掛號", "掛號", "未指定"),
    ("我要掛內科", "掛號", "內科")
]

def run_test_cases():
    print("🧪 === 開始執行 AI 語意解析測試清單 ===\n")
    passed_count = 0

    for idx, (user_text, expected_action, expected_dept) in enumerate(test_cases, 1):
        raw_reply, action, dept = parse_user_input(user_text)

        # 判斷是否符合預期結果
        is_passed = (action == expected_action) and (dept == expected_dept)
        if is_passed:
            passed_count += 1
            status = "✅ 通過"
        else:
            status = "❌ 失敗"

        print(f"#{idx} [{status}]")
        print(f" ├─ 使用者說的話: {user_text}")
        print(f" ├─ AI 原始回覆: {raw_reply}")
        print(f" ├─ 系統解讀結果: 動作 = [{action}], 對象 = [{dept}]")
        print(f" └─ 預期結果: 動作 = [{expected_action}], 對象 = [{expected_dept}]\n")

    print(f"🎉 測試完成！總共 {len(test_cases)} 句，通過 {passed_count} 句。")

# 執行測試
run_test_cases()

🧪 === 開始執行 AI 語意解析測試清單 ===

#1 [✅ 通過]
 ├─ 使用者說的話: 我想掛號內科
 ├─ AI 原始回覆: 掛號,內科
 ├─ 系統解讀結果: 動作 = [掛號], 對象 = [內科]
 └─ 預期結果: 動作 = [掛號], 對象 = [內科]

#2 [✅ 通過]
 ├─ 使用者說的話: 請幫我取消牙科掛號
 ├─ AI 原始回覆: 取消,牙科
 ├─ 系統解讀結果: 動作 = [取消], 對象 = [牙科]
 └─ 預期結果: 動作 = [取消], 對象 = [牙科]

#3 [✅ 通過]
 ├─ 使用者說的話: 請問外科還有名額嗎？
 ├─ AI 原始回覆: 查詢,外科
 ├─ 系統解讀結果: 動作 = [查詢], 對象 = [外科]
 └─ 預期結果: 動作 = [查詢], 對象 = [外科]

#4 [✅ 通過]
 ├─ 使用者說的話: 目前各科還剩下多少名額？
 ├─ AI 原始回覆: 查詢,未指定
 ├─ 系統解讀結果: 動作 = [查詢], 對象 = [未指定]
 └─ 預期結果: 動作 = [查詢], 對象 = [未指定]

#5 [✅ 通過]
 ├─ 使用者說的話: 我最近肚子很痛，應該掛哪一科？
 ├─ AI 原始回覆: 諮詢門診,未指定
 ├─ 系統解讀結果: 動作 = [諮詢門診], 對象 = [未指定]
 └─ 預期結果: 動作 = [諮詢門診], 對象 = [未指定]

#6 [✅ 通過]
 ├─ 使用者說的話: 我想預約眼科門診
 ├─ AI 原始回覆: 掛號,未知
 ├─ 系統解讀結果: 動作 = [掛號], 對象 = [未知]
 └─ 預期結果: 動作 = [掛號], 對象 = [未知]

#7 [✅ 通過]
 ├─ 使用者說的話: 我要掛號
 ├─ AI 原始回覆: 掛號,未指定
 ├─ 系統解讀結果: 動作 = [掛號], 對象 = [未指定]
 └─ 預期結果: 動作 = [掛號], 對象 = [未指定]

#8 [✅ 通過]
 ├─ 使用者說的話: 我要掛內科
 ├─ AI 原始回覆: 掛號,內科
 ├─ 系統解讀結果: 動作 = [掛號], 對象 = [內科]
 └─ 預期結果: 動作 = [掛號], 對象 = [內科]

🎉 測試完成！總共 8 句，通過 8 句。


In [8]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）import sqlite3

def query_availability(dept_name="未指定"):
    """查詢剩餘名額"""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    try:
        if dept_name in ["內科", "外科", "牙科"]:
            cursor.execute("SELECT daily_limit, registered_count FROM departments WHERE dept_name = ?", (dept_name,))
            # 7. fetchone() 之後一定要先檢查是否為 None
            row = cursor.fetchone()
            if row is not None:
                remaining = row[0] - row[1]
                return f"【{dept_name}】目前剩餘名額：{remaining} / {row[0]} 人。"
            else:
                return f"❌ 找不到【{dept_name}】的相關資訊。"
        else:
            cursor.execute("SELECT dept_name, daily_limit, registered_count FROM departments")
            rows = cursor.fetchall()
            msg = "🏥 【各科門診剩餘名額統計】\n"
            for row in rows:
                remaining = row[1] - row[2]
                msg += f"• {row[0]}：剩餘 {remaining} 人 (上限 {row[1]} 人)\n"
            return msg.strip()
    finally:
        conn.close()

def register_appointment(patient_name, dept_name):
    """辦理門診掛號"""
    if dept_name not in ["內科", "外科", "牙科"]:
        return "❌ 掛號失敗：本院僅開放「內科」、「外科」、「牙科」掛號服務。"

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    try:
        # 檢查名額
        cursor.execute("SELECT daily_limit, registered_count FROM departments WHERE dept_name = ?", (dept_name,))
        row = cursor.fetchone()

        # 7. 檢查是否為 None
        if row is None:
            return f"❌ 系統錯誤：找不到【{dept_name}】科別。"

        daily_limit, registered_count = row[0], row[1]

        if registered_count >= daily_limit:
            return f"❌ 掛號失敗：【{dept_name}】今日名額已滿（上限 {daily_limit} 人）！"

        # 執行掛號寫入與數量加 1
        cursor.execute("INSERT INTO registrations (patient_name, dept_name, status) VALUES (?, ?, ?)",
                       (patient_name, dept_name, "已掛號"))
        cursor.execute("UPDATE departments SET registered_count = registered_count + 1 WHERE dept_name = ?",
                       (dept_name,))
        conn.commit()
        return f"✅ 掛號成功！【{patient_name}】已成功預約【{dept_name}】。"
    finally:
        conn.close()

def cancel_appointment(patient_name, dept_name):
    """取消門診掛號"""
    if dept_name not in ["內科", "外科", "牙科"]:
        return "❌ 取消失敗：請指定要取消的有效科別（內科/外科/牙科）。"

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    try:
        # 檢查是否有已掛號紀錄
        cursor.execute("""
            SELECT id FROM registrations
            WHERE patient_name = ? AND dept_name = ? AND status = '已掛號'
            ORDER BY id DESC LIMIT 1
        """, (patient_name, dept_name))
        row = cursor.fetchone()

        # 7. 檢查是否為 None
        if row is None:
            return f"❌ 取消失敗：查無【{patient_name}】在【{dept_name}】的有效掛號紀錄。"

        reg_id = row[0]
        # 更新狀態並扣減名額
        cursor.execute("UPDATE registrations SET status = '已取消' WHERE id = ?", (reg_id,))
        cursor.execute("UPDATE departments SET registered_count = registered_count - 1 WHERE dept_name = ?", (dept_name,))
        conn.commit()
        return f"✅ 取消成功！【{patient_name}】已取消【{dept_name}】的門診掛號。"
    finally:
        conn.close()

In [9]:
# ============================================================
# 第 7 格：主流程與完整流程測試
# 提醒：最後呼叫 show_tables() 確認資料庫有變化
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
def process_request(user_input, patient_name="石詠澤"):
    """
    主要服務流程整合門頭
    """
    raw_reply, action, dept = parse_user_input(user_input)

    # 業務邏輯路由
    if action == "諮詢門診":
        return ("💡 【安全提醒】本系統為純行政掛號系統，不提供任何醫療建議、病況診斷或科別推薦。\n"
                "若不確定應看哪一科，請直接說明欲掛號之科別（內科/外科/牙科），或親洽醫院現場櫃檯/總機諮詢。")

    elif action == "查詢":
        return query_availability(dept)

    elif action == "掛號":
        if dept == "未指定":
            return "❓ 請說明您具體要掛號的科別（例如：我要掛內科）。"
        elif dept == "未知":
            return "❌ 本院目前僅提供「內科」、「外科」、「牙科」之掛號服務。"
        else:
            return register_appointment(patient_name, dept)

    elif action == "取消":
        if dept in ["未指定", "未知"]:
            return "❓ 請說明您要取消哪一個科別的掛號（例如：取消牙科掛號）。"
        else:
            return cancel_appointment(patient_name, dept)

    else:
        return "❓ 抱歉，系統無法理解您的請求。請說明您要掛號、取消或查詢名額（僅限內科、外科、牙科）。"

# 重新初始化資料庫進行邊界條件測試
init_db()

print("\n🧪 === 邊界條件測試：連續掛號內科至名額全滿 (5人) ===")
for i in range(1, 6):
    res = process_request("我要掛內科", patient_name=f"測試病患{i}")
    print(f"第 {i} 次掛號：{res}")

print("\n🧪 === 邊界條件測試：第 6 次嘗試掛號內科（預期失敗） ===")
res_overflow = process_request("我要掛內科", patient_name="測試病患6")
print(f"第 6 次掛號：{res_overflow}")

print("\n🧪 === 安全規定測試：詢問病症 ===")
res_safety = process_request("我最近一直頭痛胃痛，應該掛哪一科？")
print(f"詢問症狀回應：\n{res_safety}")

✅ 資料庫初始化完成！科別名額表與掛號紀錄表已重建。

🧪 === 邊界條件測試：連續掛號內科至名額全滿 (5人) ===
第 1 次掛號：✅ 掛號成功！【測試病患1】已成功預約【內科】。
第 2 次掛號：✅ 掛號成功！【測試病患2】已成功預約【內科】。
第 3 次掛號：✅ 掛號成功！【測試病患3】已成功預約【內科】。
第 4 次掛號：✅ 掛號成功！【測試病患4】已成功預約【內科】。
第 5 次掛號：✅ 掛號成功！【測試病患5】已成功預約【內科】。

🧪 === 邊界條件測試：第 6 次嘗試掛號內科（預期失敗） ===
第 6 次掛號：❌ 掛號失敗：【內科】今日名額已滿（上限 5 人）！

🧪 === 安全規定測試：詢問病症 ===
詢問症狀回應：
💡 【安全提醒】本系統為純行政掛號系統，不提供任何醫療建議、病況診斷或科別推薦。
若不確定應看哪一科，請直接說明欲掛號之科別（內科/外科/牙科），或親洽醫院現場櫃檯/總機諮詢。


In [ ]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：放在最後一格；GitHub 上看不到網頁，批改看的是第 5、7 格的結果
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
import gradio as gr
import sqlite3
import pandas as pd

def get_db_tables_html():
    """取得當前資料表現況的 HTML 格式"""
    conn = sqlite3.connect(DB_NAME)
    try:
        df_dept = pd.read_sql_query("SELECT * FROM departments", conn)
        df_reg = pd.read_sql_query("SELECT * FROM registrations", conn)

        dept_html = df_dept.to_html(classes="table table-striped", index=False)
        reg_html = df_reg.to_html(classes="table table-striped", index=False)

        return f"<h4>📊 科別名額表 (departments)</h4>{dept_html}<br><h4>📋 掛號紀錄表 (registrations)</h4>{reg_html}"
    finally:
        conn.close()

def gradio_interface(patient_name, user_input):
    """Gradio 互動響應函式"""
    if not patient_name.strip():
        patient_name = "匿名民眾"
    if not user_input.strip():
        return "請輸入您的需求說明！", get_db_tables_html()

    # 執行主流程
    response_msg = process_request(user_input, patient_name)

    # 取得最新資料庫畫面
    tables_html = get_db_tables_html()

    return response_msg, tables_html

# 建立 Gradio Web 介面
with gr.Blocks(title="醫院門診簡化掛號系統") as demo:
    gr.Markdown("# 🏥 醫院門診簡化掛號系統")
    gr.Markdown("**學號：411312422 | 姓名：石詠澤**")
    gr.Markdown("> 💡 **系統說明**：本院僅提供 **內科、外科、牙科** 之掛號與查詢。依據安全規定，本系統**嚴禁提供任何醫療建議與診斷推薦**。")

    with gr.Row():
        with gr.Column(scale=1):
            name_input = gr.Textbox(label="請輸入您的姓名", value="石詠澤", placeholder="請輸入姓名")
            user_input = gr.Textbox(label="請輸入您的需求（自然語言）", placeholder="例如：我要掛內科 / 請幫我取消牙科 / 外科還有名額嗎？", lines=2)
            submit_btn = gr.Button("傳送請求", variant="primary")

            output_box = gr.Textbox(label="系統對話回應", lines=5)

        with gr.Column(scale=1):
            table_display = gr.HTML(label="資料庫即時狀態", value=get_db_tables_html())

    # 設定按鈕事件觸發
    submit_btn.click(
        fn=gradio_interface,
        inputs=[name_input, user_input],
        outputs=[output_box, table_display]
    )

# 啟動 Gradio 服務
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1b4fa8cbb13a18351e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ✅ 交件前檢查清單（每一項都要打勾）

- [ ] 程式裡**沒有**任何 `gsk_` 開頭的金鑰，也沒有 print 出金鑰
- [ ] 規格書第 1～7 節都填完，測試清單至少 8 句
- [ ] 選單「執行階段 → 重新啟動並執行全部」，**全部跑完沒有紅色錯誤**
- [ ] 每一格左邊的執行編號是**連續遞增**的（1、2、3…）
- [ ] 選單「編輯 → 筆記本設定」，**沒有**勾選「儲存筆記本時忽略程式碼儲存格輸出內容」
- [ ] 選單「檔案 → 在 GitHub 中儲存副本」，存到**公開（Public）**的 repo
- [ ] 用無痕視窗打開 GitHub 上的檔案，確認看得到程式和執行結果